In [ ]:
# -- Cell 1 -- rclone + Drive. Same pattern as the GCPNet notebook: Kaggle is a# fresh machine every session, so rclone is installed and configured each time.# Requires: Settings -> Internet ON, and the RCLONE_DRIVE_TOKEN secret.# Internet is needed for a second reason here -- the teacher weights are pulled# from HuggingFace at runtime and run through the AUTHORS' own training script.import os, subprocessr = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)if r.returncode not in (0, 3):          # 3 = already installed and current    raise RuntimeError(f"rclone install failed (exit {r.returncode})")from kaggle_secrets import UserSecretsClienttoken = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")os.makedirs("/root/.config/rclone", exist_ok=True)with open("/root/.config/rclone/rclone.conf", "w") as f:    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")REMOTE = "drive:Distillation"                 # MyDrive/Distillationout = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)print(out.stdout or out.stderr)assert out.returncode == 0, "cannot see " + REMOTE + " -- check the token and the folder name"

In [ ]:
# -- Cell 2 -- pull the authors' code + data down from Drive.# The folder is uploaded uncompressed, so this is a plain copy -- nothing to extract.## It copies only their_repo/data and their_repo/training, NOT the whole repo.# their_repo is 468 files / 490 MB, but 401 of those files (452 MB) are# figure_generation/, which is figure CSVs this run never reads. Drive's API# charges per file, so pulling 53 files instead of 468 is the difference between# seconds and a long wait. paper/ is skipped entirely; models/ comes down in# Cell 4, which needs only the one snapshot folder rather than the whole cache.import globPROJ = "/kaggle/working/project"REPO = PROJ + "/their_repo"os.makedirs(REPO, exist_ok=True)FLAGS = "--transfers 16 --checkers 16 -P"for sub in ("data", "training"):    if os.path.isdir(REPO + "/" + sub):        print(sub, ": already present"); continue    subprocess.run("rclone copy %s/their_repo/%s %s/%s %s"                   % (REMOTE, sub, REPO, sub, FLAGS), shell=True, check=True)# The one script we run, and the data it reads. Both must exist or nothing else matters.TRAIN_PY = REPO + "/training/02_classification_benchmarks_training_code/scripts/classification_finetuning_v2.py"DATA_DIR = REPO + "/data"for p in (TRAIN_PY, DATA_DIR + "/CellPPD_train.csv", DATA_DIR + "/CellPPD_test.csv"):    assert os.path.exists(p), "missing: " + p    print("ok", p.replace(PROJ + "/", ""))import pandas as pdtr = pd.read_csv(DATA_DIR + "/CellPPD_train.csv")te = pd.read_csv(DATA_DIR + "/CellPPD_test.csv")print("\ntrain %s  %s" % (tr.shape, dict(tr.label.value_counts())))print("test  %s  %s" % (te.shape, dict(te.label.value_counts())))print("\nNote: there is no CellPPD_val.csv, so their script takes its 5-fold CV")print("branch automatically -- which is exactly what produced the published numbers.")

In [ ]:
# -- Cell 3 -- dependencies. This is the cell most likely to fail, so it checks# itself rather than letting Cell 6 die 20 minutes in.## The hard requirement is transformers 5.x. Line 24 of their script does#     transformers.tokenization_utils_tokenizers.TokenizersBackend# at import time, and that module does not exist in transformers 4.x, which is# what Kaggle ships. lightning (not pytorch-lightning -- different import name)# and peft are not preinstalled at all.subprocess.run('pip install -q -U "transformers>=5.0" "lightning>=2.4" "peft>=0.17" tokenizers',               shell=True, check=True)# peft >=0.17 dispatches every LoRA layer through is_torchao_available(), which# RAISES rather than returning False when torchao is installed but older than# 0.16.0 -- and Kaggle ships torchao 0.10.0. Nothing here quantizes anything, so# removing torchao makes that check return False and the dispatcher fall through# to the normal nn.Linear path. Upgrading torchao instead would drag in a new# torch build, which is a far bigger change than deleting a package we never use.subprocess.run("pip uninstall -y -q torchao", shell=True)from peft.import_utils import is_torchao_availableassert is_torchao_available() is False, "torchao still on the path -- LoRA will fail"print("torchao removed; peft LoRA dispatch clear")import importlib, json, torchimport numpy as npfor m in ("transformers", "lightning", "peft", "tokenizers", "sklearn"):    mod = importlib.import_module(m)    print("%-14s %s" % (m, getattr(mod, "__version__", "?")))import transformersmod = importlib.import_module("transformers.tokenization_utils_tokenizers")assert hasattr(mod, "TokenizersBackend"), \    "transformers too old -- their script imports TokenizersBackend at module level"print("\nTokenizersBackend import path OK")print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),      "| GPUs:", torch.cuda.device_count())for i in range(torch.cuda.device_count()):    p = torch.cuda.get_device_properties(i)    print("   cuda:%d  %s  %.0f GB" % (i, p.name, p.total_memory / 1e9))assert torch.cuda.is_available(), "their script hardcodes accelerator='cuda'"

In [ ]:
# -- Cell 4 -- teacher weights, pulled from Drive (they are already up there).## On Drive they sit in HuggingFace cache layout:#     models/models--aaronfeller--<variant>/snapshots/<sha>/# We copy the snapshot contents into a plain folder whose BASENAME IS THE VARIANT# NAME. That is load-bearing: their script derives the output filename from# model_name.split("/")[-1], so a folder named peptideclm-2-mlm-large produces# CellPPD_peptideclm-2-mlm-large_results.csv -- byte-identical naming to the# artifacts shipped in their repo. A path like /kaggle/working/models/mtr would# silently rename every output file.## from_pretrained() takes a local directory just as happily as a repo id, and# trust_remote_code picks up config.py / ChemPepMTR.py sitting next to the# weights. So nothing downstream changes: --model_name simply gets a path.VARIANTS = ["peptideclm-2-mlm-large"]# This is the variant the paper quotes in section 2.4: its mean test MCC over the# three seeds is (0.8760 + 0.8780 + 0.8711) / 3 = 0.8750, i.e. the "MCC of 0.875"# claim. To also run the MTR row, add "peptideclm-2-mtr-large" (also on Drive).# That roughly doubles the wall time in Cell 6.NEEDED = ["config.json", "config.py", "ChemPepMTR.py", "model.safetensors",          "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json"]MODELS = []for v in VARIANTS:    dest = "/kaggle/working/models/" + v    snaps_remote = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, v)    if not os.path.exists(dest + "/model.safetensors"):        # The snapshot folder is named by commit sha; find it rather than hardcode it.        r = subprocess.run("rclone lsf " + snaps_remote, shell=True,                           capture_output=True, text=True)        snaps = [x.strip().rstrip("/") for x in r.stdout.split() if x.strip()]        assert len(snaps) == 1, "expected one snapshot under %s, found %s" % (snaps_remote, snaps)        os.makedirs(dest, exist_ok=True)        subprocess.run("rclone copy %s/%s %s --transfers 8 -P" % (snaps_remote, snaps[0], dest),                       shell=True, check=True)    missing = [f for f in NEEDED if not os.path.exists(dest + "/" + f)]    assert not missing, "incomplete upload for %s -- missing %s" % (v, missing)    MODELS.append(dest)# Verify the weights arrived intact by reading the safetensors header only -- no# 1.3 GB load. A truncated Drive transfer shows up here as a bad header or a# wrong block count, not as a confusing failure inside fold 1.import structfor path in MODELS:    f = path + "/model.safetensors"    with open(f, "rb") as fh:        n = struct.unpack("<Q", fh.read(8))[0]        hdr = json.loads(fh.read(n))    hdr.pop("__metadata__", None)    nparam = sum(int(np.prod(t["shape"])) for t in hdr.values())    nblocks = len({k.split("blocks.")[1].split(".")[0] for k in hdr if "blocks." in k})    print("%-28s %.1f GB on disk | %.1fM params | %d blocks"          % (os.path.basename(path), os.path.getsize(f) / 1e9, nparam / 1e6, nblocks))    assert nblocks == 32 and 330 < nparam / 1e6 < 340, "not the 337M large model"print("\nteacher(s) ready:", MODELS)

In [ ]:
# -- Cell 5 -- smoke test. Do NOT skip this: it is 3 minutes against ~1.5 hours,# and it exercises the two things in their script that are most likely to break# on a machine that is not theirs:##   1. devices=[int(args.gpu_index)] -- gpu_index defaults to None, so omitting#      the flag is a TypeError. We always pass it.#   2. PeptideModel.load_from_checkpoint(...) after each fold. torch 2.6 flipped#      torch.load's weights_only default, which is why their file carries an#      add_safe_globals() patch at the top. If that patch is stale against the#      installed versions, it dies HERE, at the end of fold 1, not at fold 5 of#      seed 3.## Their code is not modified. The subsample is done by pointing --data_dir at a# small copy of the CSVs, which is a flag they already expose.SMOKE = "/kaggle/working/smoke_data"os.makedirs(SMOKE, exist_ok=True)tr.groupby("label", group_keys=False).sample(32, random_state=0) \  .sample(frac=1, random_state=0).to_csv(SMOKE + "/CellPPD_train.csv", index=False)te.groupby("label", group_keys=False).sample(16, random_state=0) \  .to_csv(SMOKE + "/CellPPD_test.csv", index=False)print("smoke set: 64 train / 32 test\n")r = subprocess.run(    ["python", "-u", TRAIN_PY,     "--dataset", "CellPPD", "--gpu", "0", "--gpu_index", "0",     "--model_name", MODELS[0],     "--data_dir", SMOKE,     "--save_path", "/kaggle/working/smoke_out",     "--output_dir", "/kaggle/working/smoke_out",     "--log_dir", "/tmp/smoke_logs",     "--seed", "0"],    cwd=REPO, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))assert r.returncode == 0, "smoke run failed -- fix this before Cell 6"smoke_csv = glob.glob("/kaggle/working/smoke_out/*_results.csv")print("\nsmoke output:", smoke_csv)print(pd.read_csv(smoke_csv[0]).head())print("\nPipeline runs end to end. Numbers here are meaningless (64 molecules).")# The smoke run is cheap in compute but NOT in disk: their ModelCheckpoint saves# the whole 337M module once per fold, so even 64 molecules leave 5 x 1.4 GB# behind. Delete it now or the real run has 7 GB less room to work with.import shutilshutil.rmtree("/tmp/smoke_logs", ignore_errors=True)shutil.rmtree("/kaggle/working/smoke_out", ignore_errors=True)shutil.rmtree("/kaggle/working/smoke_data", ignore_errors=True)print("smoke artifacts purged")

In [ ]:
# -- Cell 5b -- background sync every 10 min. Insurance only: Cell 8 still does# the authoritative final sync. Without this, a session that dies at hour 2# loses every prediction, and this run has already outlived two sessions.# Mirrors only the small prediction CSVs -- never the checkpoints.os.makedirs("/kaggle/working/results", exist_ok=True)subprocess.Popen(    "while true; do "    "  rclone copy /kaggle/working/results "    "    drive:Distillation/results/cellppd_repro/results "    "    --drive-chunk-size 64M >> /tmp/rclone_sync.log 2>&1; "    "  sleep 600; "    "done",    shell=True)print("sync started -> results/ mirrors to Drive every 10 minutes")

In [ ]:
# -- Cell 6 -- the real runs. Three seeds, matching the seeds in the authors'# own run_metadata.json (101 / 202 / 303) so the results are directly comparable# to the artifacts shipped in their repo.## Every argument below is copied from their recorded command; the only changes# are paths and the GPU index. In particular batch_size is NOT passed, so it# takes the script default of 16 -- as theirs did.## Checkpoints go to /tmp, not /kaggle/working. Their ModelCheckpoint saves the# whole LightningModule once per fold -- 15 files x ~1.4 GB = ~21 GB, which# overruns the 20 GB /kaggle/working quota. /tmp sits on the bigger disk.## One process per GPU, each pinned with CUDA_VISIBLE_DEVICES so that cuda:0 is# always correct inside the process. On a T4 x2 the three seeds take about two# scheduling rounds; expect ~25-35 min per seed for one model.import time, itertools, json, shutilOUT = "/kaggle/working/results"LOGS = "/tmp/logs"          # NOT /kaggle/working -- see note belowSEEDS = [101, 202, 303]NGPU = torch.cuda.device_count()ADAPT = "/kaggle/working/adapters"os.makedirs(ADAPT, exist_ok=True)def slim_and_purge(variant, seed):    """Keep ~13 MB per fold, delete ~7 GB per seed.    Their ModelCheckpoint saves the entire LightningModule once per fold, so a    3-seed run leaves 15 x 1.4 GB on disk and overruns the container. Only the    LoRA matrices and the 2-layer head were ever trained; the rest is 15 frozen    copies of the same teacher trunk. This runs only after the seed's process has    EXITED -- deleting while it is alive would race the load_from_checkpoint that    their script does at the end of every fold.    """    d = "%s/%s/seed_%d" % (LOGS, variant, seed)    for ck in sorted(glob.glob(d + "/**/*.ckpt", recursive=True)):        try:            sd = torch.load(ck, map_location="cpu", weights_only=False)["state_dict"]        except Exception as e:            print("   could not read %s: %s" % (ck, e)); continue        keep = {k: v for k, v in sd.items()                if "lora_" in k or k.startswith("fc1.") or k.startswith("fc2.")}        if keep:            tag = os.path.basename(ck).replace(".ckpt", "")            torch.save(keep, "%s/%s__seed%d__%s.pt" % (ADAPT, variant, seed, tag))    freed = shutil.disk_usage("/tmp").free    shutil.rmtree(d, ignore_errors=True)    print("   purged checkpoints for %s seed %d (+%.1f GB free)"          % (variant, seed, (shutil.disk_usage("/tmp").free - freed) / 1e9))jobs = []for repo, seed in itertools.product(MODELS, SEEDS):    variant = repo.split("/")[-1]    run_dir = "%s/cellppd/%s/seed_%d" % (OUT, variant, seed)    os.makedirs(run_dir, exist_ok=True)    cmd = ["python", "-u", TRAIN_PY,           "--dataset", "CellPPD", "--gpu", "0", "--gpu_index", "0",           "--model_name", repo,           "--data_dir", DATA_DIR,           "--save_path", run_dir, "--output_dir", run_dir,           "--log_dir", "%s/%s/seed_%d" % (LOGS, variant, seed),           "--seed", str(seed)]    json.dump({"model_name": repo, "seed": seed, "command": cmd},              open(run_dir + "/run_metadata.json", "w"), indent=2)    jobs.append((variant, seed, run_dir, cmd))print("%d jobs over %d GPU(s)\n" % (len(jobs), NGPU))queue, running, t0 = list(jobs), [], time.time()free = list(range(NGPU))          # which cards are actually idlewhile queue or running:    while queue and free:        variant, seed, run_dir, cmd = queue.pop(0)        gpu = str(free.pop(0))        logp = "/tmp/%s_seed%d.log" % (variant, seed)        p = subprocess.Popen(cmd, cwd=REPO, stdout=open(logp, "w"),                             stderr=subprocess.STDOUT,                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))        running.append((variant, seed, run_dir, p, logp, int(gpu)))        print("[%6.1f min] launched %s seed %d on GPU %s" %              ((time.time() - t0) / 60, variant, seed, gpu))    time.sleep(120)    still = []    for variant, seed, run_dir, p, logp, gid in running:        if p.poll() is None:            last = [l.rstrip() for l in open(logp) if "Epoch" in l or "it/s" in l]            print("[%6.1f min] %s seed %d running | %s" %                  ((time.time() - t0) / 60, variant, seed, (last[-1][-90:] if last else "...")))            still.append((variant, seed, run_dir, p, logp, gid))        else:            free.append(gid)            slim_and_purge(variant, seed)            ok = p.returncode == 0 and glob.glob(run_dir + "/*_results.csv")            print("[%6.1f min] %s seed %d FINISHED rc=%d %s" %                  ((time.time() - t0) / 60, variant, seed, p.returncode,                   "OK" if ok else "<-- NO OUTPUT, see " + logp))    running = stillprint("\nall runs done in %.1f min" % ((time.time() - t0) / 60))

In [ ]:
# -- Cell 7 -- score it. Their script writes raw predictions only; the code that# turned those into the MCCs in the paper was never released. This reproduces it.## The aggregation was reverse-engineered from the artifacts shipped in their repo# (figure_generation/results/runs_LoRA_highrank/cellppd_all.csv, which is the file# the paper's Fig 4D notebook actually reads): each of the 5 folds predicts on the# FULL 300-row test set, and the fold predictions are ensembled by mean logit and# thresholded at 0.## CALIBRATION, measured against all 9 shipped runs (3 variants x 3 seeds): NO# aggregation reproduces cellppd_all.csv exactly. mean-logit lands 0.009 off on# average and 0.020 worst case; median and mean-sigmoid are equivalent. That# residual means their published metrics and their shipped predictions came from# slightly different runs. So judge this reproduction at |delta| < 0.02, NOT# < 0.01 -- a delta near +0.01 is the aggregation, not your run.import numpy as npfrom sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score, f1_scoreREFERENCE = {                      # their runs_LoRA_highrank/cellppd_all.csv, test MCC    "peptideclm-2-mtr-large":    {101: 0.8618, 202: 0.8686, 303: 0.8696},    "peptideclm-2-mlm-large":    {101: 0.8760, 202: 0.8780, 303: 0.8711},    "peptideclm-2-hybrid-large": {101: 0.8626, 202: 0.8605, 303: 0.8560},}rows = []for f in sorted(glob.glob(OUT + "/cellppd/*/seed_*/*_results.csv")):    seed_dir = os.path.dirname(f)                       # .../cellppd/<variant>/seed_101    seed = int(os.path.basename(seed_dir).split("_")[1])    variant = os.path.basename(os.path.dirname(seed_dir))    d = pd.read_csv(f)    d["i"] = d.groupby("fold").cumcount()          # align the 5 copies of the test set    g = d.groupby("i")    y = g.true_label.first().values    p = g.predicted_label.mean().values            # ensemble = mean logit    per_fold = [matthews_corrcoef(gg.true_label, (gg.predicted_label > 0).astype(int))                for _, gg in d.groupby("fold")]    ref = REFERENCE.get(variant, {}).get(seed, np.nan)    mcc = matthews_corrcoef(y, (p > 0).astype(int))    rows.append(dict(model=variant, seed=seed, mcc=mcc, ref=ref, delta=mcc - ref,                     auroc=roc_auc_score(y, p), acc=accuracy_score(y, (p > 0).astype(int)),                     f1=f1_score(y, (p > 0).astype(int)),                     worst_fold=min(per_fold), folds=np.round(per_fold, 3)))assert rows, "no *_results.csv found under %s -- every seed failed, check /tmp/*.log" % OUTres = pd.DataFrame(rows).sort_values(["model", "seed"])pd.set_option("display.width", 200)print(res[["model", "seed", "mcc", "ref", "delta", "auroc", "acc", "f1"]].to_string(index=False))print("\nper-fold MCC (a 0.000 fold means that fold's head collapsed --")print("it happened in the authors' own mlm-large seed 101, twice):")for _, r in res.iterrows():    print("  %-28s seed %d  %s" % (r.model, r.seed, r.folds))print("\n--- reproduction summary ---")for m, g in res.groupby("model"):    print("%-28s ours %.4f +/- %.4f   theirs %.4f   delta %+.4f" %          (m, g.mcc.mean(), g.mcc.std(), g.ref.mean(), g.mcc.mean() - g.ref.mean()))print("\n|delta| < 0.02 is a successful reproduction. Two sources of slack:")print("their aggregation is not exactly recoverable (0.009 mean, 0.020 max,")print("measured on their own shipped predictions), and seed_everything() does")print("not make CUDA reductions deterministic across different GPUs.")print("A delta near +0.01 is EXPECTED and is not evidence of a problem; what")print("would matter is a large negative delta or a collapsed ensemble.")res.drop(columns=["folds"]).to_csv(OUT + "/cellppd_metrics.csv", index=False)

In [ ]:
# -- Cell 8 -- final authoritative sync. The background loop in Cell 5b only# fires every 10 minutes, so force one now or the last seed dies with the session.## Checkpoints were already slimmed to adapters and deleted per-seed in Cell 6# (they would otherwise be 21 GB of mostly-identical frozen trunk), so there is# nothing left here but small files.DEST = REMOTE + "/results/cellppd_repro"n_ad = len(glob.glob(ADAPT + "/*.pt"))sz_ad = sum(os.path.getsize(f) for f in glob.glob(ADAPT + "/*.pt")) / 1e6print("uploading: %d prediction files, %d adapters (%.1f MB)"      % (len(glob.glob(OUT + "/**/*_results.csv", recursive=True)), n_ad, sz_ad))subprocess.run("rclone copy %s %s/results --drive-chunk-size 64M -P" % (OUT, DEST),               shell=True, check=True)subprocess.run("rclone copy %s %s/adapters --drive-chunk-size 64M -P" % (ADAPT, DEST),               shell=True, check=True)print("\nuploaded to " + DEST)print(subprocess.run("rclone lsf -R " + DEST, shell=True,                     capture_output=True, text=True).stdout)